In [1]:
from app.config import get_async_engine
from sqlalchemy import text

In [2]:
engine = get_async_engine()

In [3]:
from psycopg import sql

tokens = ["services", "monolithic", "system", "design"]

async with engine.connect() as conn:
    res = await conn.execute(
        text("SELECT _id FROM token_list WHERE token = ANY(:tokens)"),
        {"tokens": tokens}
    )

[row for row in res]
# res.fetchall()

[(460,), (627,), (2095,)]

In [4]:
url_ids = [1, 2, 3]

In [5]:
from sherloque.utils import preprocess_text

query = "services and monolithic system design"
qtoks = await preprocess_text(query)
qtoks

['service', 'and', 'monolithic', 'system', 'design']

In [6]:
qtids = []
async with engine.connect() as conn:
    res = await conn.execute(
        text("SELECT _id FROM token_list WHERE token = ANY(:tokens)"),
        {"tokens": qtoks}
    )
    qtids = list(map(lambda x: x[0], res.fetchall()))

qtids

[107, 460, 627, 1370, 2095]

In [7]:
async with engine.connect() as conn:
    res = await conn.execute(
        text("""
        SELECT url_id, COUNT(token_id)
        FROM token_location
        WHERE token_id = ANY(:token_ids) AND url_id = ANY(:url_ids)
        GROUP BY url_id
        """),
        {"token_ids": qtids, "url_ids": url_ids}
    )

res.fetchall()

[(1, 53), (2, 69), (3, 82)]

In [8]:
async with engine.connect() as conn:
    res = await conn.execute(
        text("""
                SELECT tl.url_id, tl.token_id, MIN(tl.location)
                FROM token_location tl
                WHERE token_id = ANY(:token_ids)
                    AND url_id = ANY(:url_ids)
                GROUP BY tl.url_id, tl.token_id
                """),
        {"token_ids": qtids, "url_ids": url_ids}
    )
    
rows = [record for record in res]
rows

[(1, 107, 225),
 (1, 460, 910),
 (1, 627, 1388),
 (2, 107, 230),
 (2, 460, 472),
 (2, 627, 535),
 (2, 1370, 1938),
 (3, 107, 196),
 (3, 460, 446),
 (3, 627, 6),
 (3, 1370, 232)]

In [9]:
pos_sum_per_url = {out_row[0]: sum([in_row[2] for in_row in rows if in_row[0] == out_row[0]]) for out_row in rows}
pos_sum_per_url

{1: 2523, 2: 3175, 3: 880}